# Brain Tumor MRI Classification with Transfer Learning

**Name:** First Last

**Course / Assignment:** MT 36 — ML Pipeline Project (S45)

**Goal:** Classify brain MRI slices into one of four classes — `glioma`, `meningioma`,
`pituitary`, `notumor` — using a pretrained ImageNet backbone adapted to this dataset via
transfer learning. Two adaptation strategies are built and compared: (a) feature extraction
(frozen backbone, new head only) and (b) fine-tuning (backbone partially unfrozen at a lower
learning rate).

**Dataset:** [Brain Tumor MRI Dataset](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset)
(Kaggle, masoudnickparvar) — 7,023 T1-weighted grayscale MRI slices, pre-split into
`Training/` (5,712 images) and `Testing/` (1,311 images), each with four class subfolders.


## 1. Environment Setup

This notebook uses **PyTorch** + **torchvision** for the modeling pipeline. If a GPU (CUDA
or Apple MPS) is available it will be used automatically; otherwise it falls back to CPU
(training will just be slower).

Run the cell below first. If any package is missing, uncomment the `pip install` line.

In [ ]:
# !pip install torch torchvision scikit-learn matplotlib seaborn pillow kagglehub -q

import os
import random
import time
import copy
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    roc_curve,
    auc,
)
from sklearn.preprocessing import label_binarize

# ---- Reproducibility -------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ---- Device -----------------------------------------------------------------
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## 2. Data Acquisition

The dataset comes pre-split into `Training/` and `Testing/` folders, each with four class
subfolders (`glioma`, `meningioma`, `pituitary`, `notumor`). The cell below tries `kagglehub`
first (works out of the box in Colab/Kaggle notebooks with your Kaggle credentials configured);
if that's not available, it falls back to a manually-set local path.

**If you're running this locally:** download the dataset from Kaggle, unzip it, and set
`DATA_ROOT` to point at the folder that directly contains `Training/` and `Testing/`.

In [ ]:
DATA_ROOT = None  # set this manually to a local path if kagglehub isn't available,
                   # e.g. DATA_ROOT = "/content/brain-tumor-mri-dataset"

if DATA_ROOT is None:
    try:
        import kagglehub
        DATA_ROOT = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
        print("Downloaded via kagglehub to:", DATA_ROOT)
    except Exception as e:
        raise RuntimeError(
            "Could not auto-download via kagglehub. Download the dataset manually from "
            "https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset, "
            "unzip it, and set DATA_ROOT above to the folder containing Training/ and Testing/."
        ) from e

DATA_ROOT = Path(DATA_ROOT)

# The zip sometimes unpacks with an extra nested folder — auto-detect where
# Training/Testing actually live.
candidates = [DATA_ROOT] + [p for p in DATA_ROOT.rglob("*") if p.is_dir()]
root = None
for c in candidates:
    if (c / "Training").is_dir() and (c / "Testing").is_dir():
        root = c
        break

if root is None:
    raise RuntimeError(f"Could not find Training/Testing folders under {DATA_ROOT}")

TRAIN_DIR = root / "Training"
TEST_DIR = root / "Testing"
CLASSES = sorted([d.name for d in TRAIN_DIR.iterdir() if d.is_dir()])

print("Train dir:", TRAIN_DIR)
print("Test dir: ", TEST_DIR)
print("Classes:  ", CLASSES)

## 3. Exploratory Data Analysis

Before touching the model, let's understand the data: how balanced are the classes, what do
sample images from each class actually look like, and how consistent are image sizes/modes
across the three source datasets that got merged together.

In [ ]:
def count_per_class(folder):
    counts = {}
    for cls in CLASSES:
        counts[cls] = len(list((folder / cls).glob("*")))
    return counts

train_counts = count_per_class(TRAIN_DIR)
test_counts = count_per_class(TEST_DIR)

summary = pd.DataFrame({"train": train_counts, "test": test_counts})
summary["total"] = summary["train"] + summary["test"]
summary.loc["TOTAL"] = summary.sum()
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
summary.drop("TOTAL").plot(kind="bar", ax=axes[0], color=["#4C72B0", "#DD8452"])
axes[0].set_title("Image count per class (train vs test)")
axes[0].set_ylabel("# images")
axes[0].tick_params(axis="x", rotation=30)

(summary.drop("TOTAL")["train"] / summary.drop("TOTAL")["train"].sum() * 100).plot(
    kind="bar", ax=axes[1], color="#55A868"
)
axes[1].set_title("Training class share (%)")
axes[1].set_ylabel("% of training set")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

print("\nClass balance check: classes range from "
      f"{summary.drop('TOTAL')['train'].min()} to {summary.drop('TOTAL')['train'].max()} "
      "training images — mild imbalance, not extreme. Worth watching per-class recall later.")

Now let's actually look at the images — a handful of samples from each class.

In [ ]:
def show_sample_grid(folder, classes, n_per_class=5):
    fig, axes = plt.subplots(len(classes), n_per_class, figsize=(2.2 * n_per_class, 2.2 * len(classes)))
    for row, cls in enumerate(classes):
        files = sorted((folder / cls).glob("*"))[:n_per_class]
        for col, f in enumerate(files):
            img = Image.open(f)
            ax = axes[row, col]
            ax.imshow(img, cmap="gray")
            ax.axis("off")
            if col == 0:
                ax.set_ylabel(cls, fontsize=11)
        axes[row, 0].set_title(cls, fontsize=12, loc="left")
    plt.suptitle("Sample MRI slices per class", y=1.02, fontsize=14)
    plt.tight_layout()
    plt.show()

show_sample_grid(TRAIN_DIR, CLASSES, n_per_class=5)

Some classes look visually distinct (e.g. `notumor` scans are clean, symmetric brain
slices), while `glioma` and `meningioma` can be subtle — a small, oddly-shaped bright/dark
region is often the only difference. That visual similarity is worth keeping in mind when we
look at the confusion matrix later.

Next, let's check how consistent image sizes and color modes are, since the brief flags that
the three merged sources vary in resolution and orientation.

In [ ]:
def inspect_images(folder, classes, n_per_class=60):
    sizes, modes = [], []
    for cls in classes:
        files = sorted((folder / cls).glob("*"))[:n_per_class]
        for f in files:
            with Image.open(f) as img:
                sizes.append(img.size)
                modes.append(img.mode)
    return sizes, modes

sizes, modes = inspect_images(TRAIN_DIR, CLASSES)
widths, heights = zip(*sizes)

print("Color modes found:", Counter(modes))
print(f"Width  range: {min(widths)}–{max(widths)} (median {int(np.median(widths))})")
print(f"Height range: {min(heights)}–{max(heights)} (median {int(np.median(heights))})")

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(widths, heights, alpha=0.4, s=15)
ax.set_xlabel("width (px)")
ax.set_ylabel("height (px)")
ax.set_title("Image dimensions across a sample of training images")
plt.show()

As expected, sizes vary quite a bit since this is a merge of three different public
sources — that's an inherent property of the data, not something to "fix." We'll standardize
everything at the model-input stage instead (resize + consistent channel count), rather than
trying to pre-clean the raw files.

## 4. Data Preparation

**Backbone choice: ResNet18** (`torchvision.models.resnet18`, pretrained on ImageNet).
It's a good default for a first transfer-learning pass — small enough to fine-tune quickly on
a mid-size dataset like this one (5,712 training images), while still transferring strong
low/mid-level visual features from ImageNet.

Two preprocessing details matter here:
- **Input size:** ResNet18 expects 224×224 RGB input.
- **Grayscale → 3-channel:** these MRI slices are single-channel grayscale, but ImageNet
  backbones expect 3-channel input. We replicate the single channel three times rather than
  training a modified first conv layer from scratch — that would throw away the pretrained
  weights for the very first layer.

We also apply the standard ImageNet normalization statistics, since the pretrained weights
expect input scaled the same way the original ImageNet training data was.

**Splits:**
- `Training/` → split 85% train / 15% validation (stratified by class), used for model
  selection and hyperparameter choices.
- `Testing/` → held out completely untouched until final evaluation in Section 7.

**Augmentation:** applied only to the training split — small random rotations and horizontal
flips to add a bit of robustness to slight orientation/positioning differences between the
three source datasets, without distorting anatomy unrealistically.

In [ ]:
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Two ImageFolder views over the SAME training directory, one per transform,
# so train/val subsets get the transform appropriate to their role.
full_train_aug = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
full_train_eval = datasets.ImageFolder(TRAIN_DIR, transform=eval_transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_transform)

assert full_train_aug.classes == CLASSES

targets = np.array(full_train_aug.targets)
idx = np.arange(len(targets))
train_idx, val_idx = train_test_split(
    idx, test_size=0.15, stratify=targets, random_state=SEED
)

train_dataset = Subset(full_train_aug, train_idx)
val_dataset = Subset(full_train_eval, val_idx)  # no augmentation on validation

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print("Class-to-index mapping:", full_train_aug.class_to_idx)

In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Sanity check: pull one batch, confirm shape and that normalization looks right
xb, yb = next(iter(train_loader))
print("Batch shape:", xb.shape)  # [B, 3, 224, 224]
print("Label sample:", yb[:8].tolist())
print("Pixel value range after normalize:", xb.min().item(), "to", xb.max().item())

## 5. Model Setup — Two Transfer Learning Strategies

Both approaches start from the same pretrained ResNet18 and replace the final fully-connected
layer (originally 1000-way for ImageNet) with a new 4-way classification head.

**Approach A — Feature Extraction:** freeze every backbone parameter. Only the new head is
trained. Fast, less prone to overfitting on a modest dataset, but limited by how well
ImageNet features transfer to grayscale medical images as-is.

**Approach B — Fine-Tuning:** unfreeze the last residual block (`layer4`) plus the new head,
and train the whole thing with a much lower learning rate on the unfrozen backbone layers than
on the head (discriminative learning rates). This lets the network adapt higher-level visual
features to MRI-specific patterns, while keeping early, more generic layers (edges, textures)
frozen since those transfer well regardless of domain.

In [ ]:
def build_model(strategy="feature_extract"):
    """strategy: 'feature_extract' or 'fine_tune'"""
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    if strategy == "feature_extract":
        for param in model.parameters():
            param.requires_grad = False
    elif strategy == "fine_tune":
        for param in model.parameters():
            param.requires_grad = False
        for param in model.layer4.parameters():
            param.requires_grad = True
    else:
        raise ValueError("strategy must be 'feature_extract' or 'fine_tune'")

    # New head — always trainable
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(num_features, len(CLASSES)),
    )
    for param in model.fc.parameters():
        param.requires_grad = True

    return model.to(device)


def count_trainable(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total


for strat in ["feature_extract", "fine_tune"]:
    m = build_model(strat)
    t, total = count_trainable(m)
    print(f"{strat:16s} — trainable params: {t:,} / {total:,} ({100*t/total:.1f}%)")
    del m

## 6. Training Loop

A shared training function handles both strategies, with early stopping on validation loss
(patience of 3 epochs) so we don't overfit either approach, and so the comparison in Section 6
is fair — both get to converge, neither is cut off arbitrarily early.

In [ ]:
def train_model(model, optimizer, train_loader, val_loader, num_epochs=15, patience=3):
    criterion = nn.CrossEntropyLoss()
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0

    for epoch in range(num_epochs):
        start = time.time()

        # ---- train ----
        model.train()
        running_loss, running_correct, n = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            outputs = model(xb)
            loss = criterion(outputs, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * xb.size(0)
            running_correct += (outputs.argmax(1) == yb).sum().item()
            n += xb.size(0)

        train_loss = running_loss / n
        train_acc = running_correct / n

        # ---- validate ----
        model.eval()
        v_loss, v_correct, v_n = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                outputs = model(xb)
                loss = criterion(outputs, yb)
                v_loss += loss.item() * xb.size(0)
                v_correct += (outputs.argmax(1) == yb).sum().item()
                v_n += xb.size(0)

        val_loss = v_loss / v_n
        val_acc = v_correct / v_n

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        elapsed = time.time() - start
        print(f"Epoch {epoch+1:2d}/{num_epochs} | "
              f"train_loss {train_loss:.4f} acc {train_acc:.4f} | "
              f"val_loss {val_loss:.4f} acc {val_acc:.4f} | {elapsed:.1f}s")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1} (no val improvement for {patience} epochs)")
                break

    model.load_state_dict(best_state)
    return model, history

### Approach A — Feature Extraction

In [ ]:
model_fe = build_model("feature_extract")
optimizer_fe = optim.Adam(filter(lambda p: p.requires_grad, model_fe.parameters()), lr=1e-3)

model_fe, history_fe = train_model(
    model_fe, optimizer_fe, train_loader, val_loader, num_epochs=15, patience=3
)

### Approach B — Fine-Tuning

Discriminative learning rates: the unfrozen `layer4` backbone params get a much smaller
learning rate than the freshly-initialized head, since the backbone weights are already
close to useful and only need gentle adjustment.

In [ ]:
model_ft = build_model("fine_tune")

optimizer_ft = optim.Adam([
    {"params": model_ft.layer4.parameters(), "lr": 1e-5},
    {"params": model_ft.fc.parameters(), "lr": 1e-3},
])

model_ft, history_ft = train_model(
    model_ft, optimizer_ft, train_loader, val_loader, num_epochs=15, patience=3
)

## 7. Comparing the Two Approaches

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for h, label in [(history_fe, "Feature extraction"), (history_ft, "Fine-tuning")]:
    axes[0].plot(h["val_loss"], label=f"{label} (val)")
    axes[0].plot(h["train_loss"], linestyle="--", alpha=0.5, label=f"{label} (train)")
    axes[1].plot(h["val_acc"], label=f"{label} (val)")
    axes[1].plot(h["train_acc"], linestyle="--", alpha=0.5, label=f"{label} (train)")

axes[0].set_title("Loss over epochs")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend(fontsize=8)
axes[1].set_title("Accuracy over epochs")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy"); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

best_val_acc_fe = max(history_fe["val_acc"])
best_val_acc_ft = max(history_ft["val_acc"])
print(f"Best validation accuracy — feature extraction: {best_val_acc_fe:.4f}")
print(f"Best validation accuracy — fine-tuning:        {best_val_acc_ft:.4f}")

best_model = model_ft if best_val_acc_ft >= best_val_acc_fe else model_fe
best_name = "fine_tune" if best_val_acc_ft >= best_val_acc_fe else "feature_extract"
print(f"\nSelected for final test evaluation: {best_name}")

*Note: which approach wins can vary run to run depending on dataset size, augmentation
strength, and epoch budget — that's expected and part of why we compare both rather than
assuming fine-tuning is automatically better. Fine-tuning has more capacity to adapt to
MRI-specific texture patterns but is also more prone to overfitting on a dataset this size if
left training too long, which is exactly what the early-stopping/validation comparison above
is meant to catch.*

## 8. Final Evaluation on the Held-Out Test Set

Everything above (including model selection) only touched `Training/` data (split into
train/val). Now, for the first time, we evaluate the selected model on `Testing/` — data the
model never saw during training or tuning.

Since this is a 4-class problem, overall accuracy alone can hide a model that's strong on the
easy classes and weak on the hard ones. We look at per-class precision/recall/F1, a confusion
matrix, and per-class accuracy explicitly.

In [ ]:
def get_predictions(model, loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            outputs = model(xb)
            probs = torch.softmax(outputs, dim=1)
            preds = probs.argmax(1)
            all_preds.append(preds.cpu().numpy())
            all_labels.append(yb.numpy())
            all_probs.append(probs.cpu().numpy())
    return (
        np.concatenate(all_preds),
        np.concatenate(all_labels),
        np.concatenate(all_probs),
    )

y_pred, y_true, y_probs = get_predictions(best_model, test_loader)

test_acc = accuracy_score(y_true, y_pred)
test_f1_macro = f1_score(y_true, y_pred, average="macro")
test_f1_weighted = f1_score(y_true, y_pred, average="weighted")

print(f"Test accuracy:        {test_acc:.4f}")
print(f"Test macro F1:        {test_f1_macro:.4f}  (treats all 4 classes equally)")
print(f"Test weighted F1:     {test_f1_weighted:.4f}  (weighted by class support)")
print()
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0])
axes[0].set_title("Confusion matrix (counts)")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")

sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1])
axes[1].set_title("Confusion matrix (row-normalized = recall per class)")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")
plt.tight_layout()
plt.show()

In [ ]:
per_class_acc = cm.diagonal() / cm.sum(axis=1)
acc_df = pd.DataFrame({"class": CLASSES, "recall/accuracy": per_class_acc}).sort_values(
    "recall/accuracy"
)

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=acc_df, x="recall/accuracy", y="class", ax=ax, palette="viridis")
ax.set_xlim(0, 1)
ax.set_title("Per-class recall on test set")
plt.tight_layout()
plt.show()

acc_df

Bonus: one-vs-rest ROC curves give another view of how well-separated each class is in the model's predicted probabilities.

In [ ]:
y_true_bin = label_binarize(y_true, classes=list(range(len(CLASSES))))

fig, ax = plt.subplots(figsize=(6, 6))
for i, cls in enumerate(CLASSES):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"{cls} (AUC = {roc_auc:.3f})")

ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("One-vs-rest ROC curves (test set)")
ax.legend()
plt.tight_layout()
plt.show()

## 9. Error Analysis — What Is the Model Getting Wrong?

Aggregate metrics tell us *how much* the model is wrong; looking at actual misclassified
images tells us *why*. Let's pull out a sample of misclassified test images, grouped by the
(true, predicted) pair that occurs most often.

In [ ]:
# Identify the most common confusion pairs (excluding the diagonal)
cm_pairs = []
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        if i != j and cm[i, j] > 0:
            cm_pairs.append((CLASSES[i], CLASSES[j], cm[i, j]))

cm_pairs_df = pd.DataFrame(cm_pairs, columns=["true", "predicted", "count"]).sort_values(
    "count", ascending=False
)
print("Most common confusion pairs:")
cm_pairs_df.head(10)

In [ ]:
def show_misclassified(true_cls, pred_cls, n=6):
    true_idx = CLASSES.index(true_cls)
    pred_idx = CLASSES.index(pred_cls)
    mask = (y_true == true_idx) & (y_pred == pred_idx)
    matches = np.where(mask)[0]

    if len(matches) == 0:
        print(f"No examples where true={true_cls}, predicted={pred_cls}")
        return

    matches = matches[:n]
    fig, axes = plt.subplots(1, len(matches), figsize=(2.3 * len(matches), 2.6))
    if len(matches) == 1:
        axes = [axes]
    for ax, idx in zip(axes, matches):
        img_path, _ = test_dataset.samples[idx]
        img = Image.open(img_path)
        ax.imshow(img, cmap="gray")
        ax.axis("off")
    fig.suptitle(f"True: {true_cls}  →  Predicted: {pred_cls}", fontsize=12)
    plt.tight_layout()
    plt.show()

# Show examples for the single most common confusion pair, and a couple more
top_pairs = cm_pairs_df.head(3)
for _, row in top_pairs.iterrows():
    show_misclassified(row["true"], row["predicted"], n=6)

**Observations to fill in after running:** note here which class pairs are most
frequently confused (commonly `glioma` vs `meningioma`, since both are tumor types that can
look visually similar in a single 2D slice — location and shape are what usually distinguish
them, and a model working on 2D slices without 3D context or clinical metadata is at a
disadvantage here), and whether `notumor` is reliably kept separate from the three tumor
classes (a `notumor` slice being misclassified as a tumor type is a very different kind of
error than confusing two tumor types with each other — worth calling out based on what the
confusion matrix in Section 7 actually shows).

## 10. Write-Up

*(Fill in the bracketed parts after running the notebook — the numbers above will differ
slightly run to run due to random initialization, augmentation, and train/val split.)*

**What worked:** Transfer learning from ImageNet, despite the large domain gap between natural
photos and grayscale MRI slices, still gave a strong starting point — [best test accuracy]%
accuracy and [macro F1] macro F1 with relatively little training. The generic low-level
features (edges, textures, shapes) that ResNet18 learned from ImageNet transferred reasonably
well as a starting point even though the target domain (single-channel medical imaging) is
quite different from natural images.

**Feature extraction vs. fine-tuning:** [State which approach had the higher validation
accuracy and briefly why you think that happened — e.g. if fine-tuning won, the extra
capacity to adapt `layer4` to MRI-specific patterns likely helped it pick up on
tumor-shape/texture cues that generic ImageNet features don't directly capture; if feature
extraction won or came close, that suggests the dataset may be small enough, or visually
distinct enough from ImageNet, that fine-tuning risked overfitting rather than helping.]
**Recommendation:** [state which one you'd ship and why — usually a function of the gap in val
accuracy vs. the added training cost/overfitting risk of fine-tuning].

**Where the model is still weak:** Per Section 8's confusion matrix and Section 9's error
analysis, [name the specific class pair(s) that get confused most, and your hypothesis for
why — e.g. glioma vs. meningioma being visually similar on a single 2D slice, or a specific
class having fewer training examples than the others]. This matters clinically: this tool is
meant to *prioritize* review, not replace it, so [note whether the current error pattern would
be safe/dangerous in a triage context — e.g. is `notumor` ever confused with a tumor class,
which would be the worse kind of mistake to make in this application, versus confusing two
tumor subtypes with each other, which is lower-stakes since both would still get flagged for
review].

**What I'd try next:** ideas worth exploring beyond this notebook's scope — a stronger/larger
backbone (e.g. EfficientNet or a ViT) if compute allows, more aggressive or medically-informed
augmentation, addressing class imbalance directly (class-weighted loss or oversampling the
smaller classes), or, most importantly, framing this as a 3-class *tumor-type* problem plus a
separate *tumor vs. no-tumor* triage step, since those are two different clinical questions
being asked at once by a single 4-way classifier.
